In [ ]:
import math
from scipy.stats import poisson

In [ ]:
# Equation 1 The probability the attacker creates k blocks over a window of time T is:

# P_attacker_creates(k, T) = (aplha * lambda * T)^k * e^(-alpha * lambda * T) / k!




# Equation 2 

# P_attacker_ahead_by_i(i, T, N) = (alpha * lambda * T)^(i+N) * e^(-alpha * lambda * T) / (i + N)!




# Equation 3

# Pdouble_spend_if_accepted_right_now(T, N) = summation from i = -N to 0 of P_attacker_ahead_by_i(i, T, N)(alpha/(1-alpha))^(-i+1) + (1 - summation from i = -N to 0 of P_attacker_ahead_by_i(i, T, N)




In [ ]:
# Equation 1 The probability the attacker creates k blocks over a window of time T is:

# P_attacker_creates(k, T) = (aplha * lambda * T)^k * e^(-alpha * lambda * T) / k!

def p_attacker_creates(k: int, T_minutes: float, alpha: float, lam: float = 0.1) -> float:
    mu = alpha * lam * T_minutes
    return float(poisson.pmf(k, mu))

# or (without the possion library) 


def p_attacker_creates_direct(k: int, T_minutes: float, alpha: float, lam: float = 0.1) -> float:
    mu = alpha * lam * T_minutes 
    if k < 0:
        return 0.0
    return (mu ** k) * math.exp(-mu) / math.factorial(k)

In [ ]:
# Equation 2 

# P_attacker_ahead_by_i(i, T, N) = (alpha * lambda * T)^(i+N) * e^(-alpha * lambda * T) / (i + N)!

def p_attacker_ahead_by_i(i: int, T_minutes: float, N: int, alpha: float, lam: float = 0.1) -> float:
    mu = alpha * lam * T_minutes
    return float(poisson.pmf(i + N, mu))

# or (without the possion library) 

def p_attacker_ahead_by_i_direct(i: int, T_minutes: float, N: int, alpha: float, lam: float = 0.1) -> float:
    k = i + N
    if k < 0:
        return 0.0
    mu = alpha * lam * T_minutes
    return (mu ** k) * math.exp(-mu) / math.factorial(k)

In [ ]:
# Equation 3

# Pdouble_spend_if_accepted_right_now(T, N) = summation from i = -N to 0 of P_attacker_ahead_by_i(i, T, N)(alpha/(1-alpha))^(-i+1) + (1 - summation from i = -N to 0 of P_attacker_ahead_by_i(i, T, N)

import math

def p_attacker_ahead_by_i_direct(i: int, T_minutes: float, N: int, alpha: float, lam: float = 0.1) -> float:
    # P_attacker_ahead_by_i(i,T,N) = (αλT)^(i+N) * e^(-αλT) / (i+N)!
    k = i + N
    if k < 0:
        return 0.0
    mu = alpha * lam * T_minutes
    return (mu ** k) * math.exp(-mu) / math.factorial(k)

def p_double_spend_if_accepted_now_direct(T_minutes: float, N: int, alpha: float, lam: float = 0.1) -> float:
    if not (0.0 < alpha < 0.5):
        raise ValueError("This formula assumes attacker has < 50% hashpower (0 < alpha < 0.5).")
    if N < 0 or T_minutes < 0:
        raise ValueError("N and T must be non-negative.")

    ratio = alpha / (1.0 - alpha)

    sum_pi = 0.0
    sum_catchup = 0.0

    for i in range(-N, 1):  # i = -N ... 0
        p_i = p_attacker_ahead_by_i_direct(i, T_minutes, N, alpha, lam)
        sum_pi += p_i
        sum_catchup += p_i * (ratio ** (-i + 1))

    p_ds = sum_catchup + (1.0 - sum_pi)

    # clamp tiny floating point drift
    return max(0.0, min(1.0, p_ds))


